# Dashboard HDI Seguros Colombia
Equivalente en Python de los 7 ejercicios KNIME — genera un dashboard HTML interactivo al final.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path

DATA = Path('data')
print('Librerías cargadas')

## Tarea 1 — Lectura y exploración de datos
Equivale a: `CSV Reader → Statistics → Data Explorer`

In [ ]:
clientes      = pd.read_csv(DATA / 'clientes.csv')
polizas       = pd.read_csv(DATA / 'polizas.csv')
siniestros    = pd.read_csv(DATA / 'siniestros.csv')
primas        = pd.read_csv(DATA / 'primas_mensual.csv')

print('=== CLIENTES — primeras filas ===')
display(clientes.head())

print('\n=== Estadísticas numéricas (equivalente a Statistics) ===')
display(clientes.describe())

# Pregunta T1: ¿Cuántos clientes por segmento?
t1_segmento = clientes['segmento'].value_counts().reset_index()
t1_segmento.columns = ['segmento', 'cantidad']
print('\n=== Clientes por segmento ===')
display(t1_segmento)

## Tarea 2 — Filtrado y selección de columnas
Equivale a: `CSV Reader → Row Filter → Column Filter → Statistics`

In [ ]:
# Row Filter: estado == 'Activa'
polizas_activas = polizas[polizas['estado'] == 'Activa'].copy()

# Column Filter: solo columnas relevantes
cols_relevantes = ['id_poliza', 'id_cliente', 'tipo_seguro', 'prima_anual', 'suma_asegurada']
polizas_activas = polizas_activas[cols_relevantes]

print(f'Pólizas activas: {len(polizas_activas)} de {len(polizas)} totales')
display(polizas_activas)

# Pregunta T2: prima anual promedio pólizas activas
prima_promedio = polizas_activas['prima_anual'].mean()
print(f'\nPrima anual promedio (activas): ${prima_promedio:,.0f} COP')

## Tarea 3 — Unión de tablas (Join)
Equivale a: `CSV Reader (x2) → Joiner (Inner Join por id_cliente)`

In [ ]:
# Inner Join: polizas + clientes por id_cliente
polizas_clientes = polizas.merge(clientes[['id_cliente', 'nombre', 'apellido', 'ciudad', 'segmento']],
                                  on='id_cliente', how='inner')

print(f'Filas tras el join: {len(polizas_clientes)}')
display(polizas_clientes[['id_poliza', 'nombre', 'apellido', 'ciudad', 'tipo_seguro', 'prima_anual', 'estado']].head(10))

# Pregunta T3: ¿Qué ciudad tiene más pólizas de Automóvil activas?
t3 = (polizas_clientes
      .query("tipo_seguro == 'Automóvil' and estado == 'Activa'")
      .groupby('ciudad')['id_poliza'].count()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={'id_poliza': 'polizas_auto_activas'}))
print('\nCiudades con más pólizas de Automóvil activas:')
display(t3)

## Tarea 4 — Agrupación y agregación + siniestralidad
Equivale a: `CSV Reader → GroupBy → Math Formula`

In [ ]:
# GroupBy tipo_seguro: suma de prima, siniestros y valor siniestros
t4 = primas.groupby('tipo_seguro').agg(
    prima_emitida_COP   = ('prima_emitida_COP',   'sum'),
    siniestros_ocurridos = ('siniestros_ocurridos', 'sum'),
    valor_siniestros_COP = ('valor_siniestros_COP', 'sum')
).reset_index()

# Math Formula: siniestralidad (%)
t4['siniestralidad_pct'] = (t4['valor_siniestros_COP'] / t4['prima_emitida_COP'] * 100).round(2)

t4_sorted = t4.sort_values('siniestralidad_pct', ascending=False)
print('=== Siniestralidad por ramo ===')
display(t4_sorted)
print(f"\nRamo con mayor siniestralidad: {t4_sorted.iloc[0]['tipo_seguro']} ({t4_sorted.iloc[0]['siniestralidad_pct']}%)")

## Tarea 5 — Análisis de siniestros (Top 5)
Equivale a: `CSV Reader (x2) → Joiner → Math Formula → Sorter → Top k Row Filter`

In [ ]:
# Join siniestros + polizas por id_poliza
t5 = siniestros.merge(polizas[['id_poliza', 'tipo_seguro', 'canal_venta']], on='id_poliza', how='inner')

# Math Formula: diferencia = valor_reclamado - valor_aprobado
t5['diferencia'] = t5['valor_reclamado'] - t5['valor_aprobado']

# Sorter desc por valor_aprobado + Top 5
top5 = t5.sort_values('valor_aprobado', ascending=False).head(5)

cols_show = ['id_siniestro', 'tipo_siniestro', 'ciudad_ocurrencia',
             'tipo_seguro', 'valor_reclamado', 'valor_aprobado', 'diferencia', 'estado']
print('=== Top 5 siniestros por valor aprobado ===')
display(top5[cols_show])

# Mayor brecha
mayor_brecha = t5.sort_values('diferencia', ascending=False).iloc[0]
print(f"\nMayor brecha: {mayor_brecha['id_siniestro']} — diferencia ${mayor_brecha['diferencia']:,.0f} COP")

## Tarea 6 — Visualizaciones
Equivale a: `Bar Chart`, `Pie/Donut Chart`, `Line Plot`

In [ ]:
# --- Bar Chart: prima emitida por tipo de seguro ---
fig_bar = px.bar(
    t4_sorted, x='tipo_seguro', y='prima_emitida_COP',
    color='siniestralidad_pct',
    color_continuous_scale='RdYlGn_r',
    title='Prima Emitida por Tipo de Seguro (COP)',
    labels={'prima_emitida_COP': 'Prima Emitida (COP)', 'tipo_seguro': 'Tipo de Seguro',
            'siniestralidad_pct': 'Siniestralidad %'},
    text_auto='.3s'
)
fig_bar.update_layout(template='plotly_white')
fig_bar.show()

# --- Pie/Donut Chart: clientes por segmento ---
fig_pie = px.pie(
    t1_segmento, names='segmento', values='cantidad',
    hole=0.4,
    title='Distribución de Clientes por Segmento',
    color_discrete_map={'Basic': '#f4a261', 'Standard': '#457b9d', 'Premium': '#2a9d8f'}
)
fig_pie.update_traces(textinfo='percent+label')
fig_pie.show()

# --- Line Plot: primas Bogotá por mes y tipo de seguro ---
bogota = primas[primas['ciudad'] == 'Bogotá'].copy()
bogota['mes_label'] = bogota['mes'].apply(lambda m: ['Ene','Feb','Mar','Abr','May','Jun',
                                                       'Jul','Ago','Sep','Oct','Nov','Dic'][m-1])
bogota_agg = bogota.groupby(['mes', 'mes_label', 'tipo_seguro'])['prima_recaudada_COP'].sum().reset_index()

fig_line = px.line(
    bogota_agg, x='mes', y='prima_recaudada_COP', color='tipo_seguro',
    markers=True,
    title='Prima Recaudada en Bogotá por Mes y Tipo de Seguro',
    labels={'prima_recaudada_COP': 'Prima Recaudada (COP)', 'mes': 'Mes', 'tipo_seguro': 'Ramo'},
    category_orders={'mes': [1, 2, 3]}
)
fig_line.update_layout(template='plotly_white')
fig_line.show()

## Tarea 7 — Flujo completo con exportación
Equivale a: `CSV Reader → Joiner → Row Filter → GroupBy → Math Formula → Excel Writer`

In [ ]:
# Join polizas + clientes → filtrar Activas → GroupBy ciudad+tipo_seguro → suma prima
t7 = (polizas_clientes[polizas_clientes['estado'] == 'Activa']
      .groupby(['ciudad', 'tipo_seguro'])['prima_anual']
      .sum()
      .reset_index()
      .rename(columns={'prima_anual': 'prima_ciudad_tipo'}))

prima_total = t7['prima_ciudad_tipo'].sum()
t7['participacion_pct'] = (t7['prima_ciudad_tipo'] / prima_total * 100).round(2)
t7 = t7.sort_values('prima_ciudad_tipo', ascending=False)

print('=== Reporte de producción por ciudad y tipo de seguro ===')
display(t7)

# Pregunta T7: ciudad con mayor prima en Vida
vida = t7[t7['tipo_seguro'] == 'Vida'].sort_values('prima_ciudad_tipo', ascending=False)
print(f"\nCiudad con mayor prima en Vida: {vida.iloc[0]['ciudad']} (${vida.iloc[0]['prima_ciudad_tipo']:,.0f} COP)")

# Exportar a Excel
out_excel = DATA / 'reporte_produccion.xlsx'
t7.to_excel(out_excel, index=False)
print(f'\nArchivo exportado: {out_excel}')

## Dashboard HTML — Generación del archivo final
Crea `dashboard_hdi.html` con todas las visualizaciones en una sola página.

In [ ]:
# =====================================================================
# Construir todas las figuras del dashboard
# =====================================================================

# --- 1. KPI cards (tabla resumen) ---
kpis = {
    'Total clientes':         len(clientes),
    'Pólizas activas':        len(polizas_activas),
    'Prima activa total (M COP)': f"{polizas_activas['prima_anual'].sum()/1e6:.1f}M",
    'Siniestros registrados': len(siniestros),
    'Valor aprobado total (M COP)': f"{siniestros['valor_aprobado'].sum()/1e6:.1f}M",
    'Siniestralidad global (%)': f"{(t4['valor_siniestros_COP'].sum()/t4['prima_emitida_COP'].sum()*100):.1f}%"
}
fig_kpi = go.Figure(go.Table(
    columnwidth=[2.5, 1.5],
    header=dict(
        values=['<b>Indicador</b>', '<b>Valor</b>'],
        fill_color='#003366',
        font=dict(color='white', size=13),
        align='left'
    ),
    cells=dict(
        values=[list(kpis.keys()), list(kpis.values())],
        fill_color=[['#f0f4ff']*len(kpis), ['#ffffff']*len(kpis)],
        font=dict(size=12),
        align=['left', 'center'],
        height=32
    )
))
fig_kpi.update_layout(title='Resumen Ejecutivo — KPIs', margin=dict(t=50, b=10))

# --- 2. Bar: prima emitida por ramo ---
fig_bar2 = px.bar(
    t4_sorted, x='tipo_seguro', y='prima_emitida_COP',
    color='siniestralidad_pct', color_continuous_scale='RdYlGn_r',
    title='Prima Emitida por Ramo (COP)',
    labels={'prima_emitida_COP': 'Prima Emitida (COP)', 'tipo_seguro': 'Ramo',
            'siniestralidad_pct': 'Siniestralidad %'},
    text_auto='.3s'
)
fig_bar2.update_layout(template='plotly_white')

# --- 3. Donut: clientes por segmento ---
fig_donut = px.pie(
    t1_segmento, names='segmento', values='cantidad', hole=0.45,
    title='Clientes por Segmento',
    color='segmento',
    color_discrete_map={'Basic': '#f4a261', 'Standard': '#457b9d', 'Premium': '#2a9d8f'}
)
fig_donut.update_traces(textinfo='percent+label')
fig_donut.update_layout(template='plotly_white')

# --- 4. Line: prima Bogotá por mes ---
fig_line2 = px.line(
    bogota_agg, x='mes', y='prima_recaudada_COP', color='tipo_seguro',
    markers=True,
    title='Prima Recaudada Bogotá por Mes',
    labels={'prima_recaudada_COP': 'Prima (COP)', 'mes': 'Mes', 'tipo_seguro': 'Ramo'}
)
fig_line2.update_layout(template='plotly_white')

# --- 5. Siniestralidad por ramo (horizontal bar) ---
fig_sini = px.bar(
    t4_sorted, x='siniestralidad_pct', y='tipo_seguro', orientation='h',
    color='siniestralidad_pct', color_continuous_scale='RdYlGn_r',
    title='Siniestralidad (%) por Ramo',
    labels={'siniestralidad_pct': 'Siniestralidad (%)', 'tipo_seguro': 'Ramo'},
    text_auto='.1f'
)
fig_sini.update_layout(template='plotly_white', showlegend=False)

# --- 6. Top 5 siniestros (tabla) ---
top5_show = top5[['id_siniestro', 'tipo_siniestro', 'ciudad_ocurrencia',
                   'tipo_seguro', 'valor_reclamado', 'valor_aprobado', 'diferencia', 'estado']].copy()
for col in ['valor_reclamado', 'valor_aprobado', 'diferencia']:
    top5_show[col] = top5_show[col].apply(lambda x: f'${x:,.0f}')

fig_top5 = go.Figure(go.Table(
    header=dict(
        values=[f'<b>{c}</b>' for c in top5_show.columns],
        fill_color='#003366', font=dict(color='white', size=11), align='center'
    ),
    cells=dict(
        values=[top5_show[c].tolist() for c in top5_show.columns],
        fill_color=[['#f0f4ff', '#ffffff'] * 3],
        font=dict(size=11), align='center', height=28
    )
))
fig_top5.update_layout(title='Top 5 Siniestros por Valor Aprobado', margin=dict(t=50, b=10))

# --- 7. Producción por ciudad y ramo (stacked bar) ---
fig_prod = px.bar(
    t7, x='ciudad', y='prima_ciudad_tipo', color='tipo_seguro',
    title='Prima Activa por Ciudad y Tipo de Seguro',
    labels={'prima_ciudad_tipo': 'Prima Anual (COP)', 'ciudad': 'Ciudad', 'tipo_seguro': 'Ramo'},
    barmode='stack', text_auto='.2s'
)
fig_prod.update_layout(template='plotly_white')

# --- 8. Canal de venta por ciudad ---
canal_ciudad = primas.groupby(['ciudad', 'canal_venta'])['prima_emitida_COP'].sum().reset_index()
fig_canal = px.bar(
    canal_ciudad, x='ciudad', y='prima_emitida_COP', color='canal_venta',
    title='Prima Emitida por Ciudad y Canal de Venta',
    labels={'prima_emitida_COP': 'Prima Emitida (COP)', 'ciudad': 'Ciudad', 'canal_venta': 'Canal'},
    barmode='group', text_auto='.2s'
)
fig_canal.update_layout(template='plotly_white')

print('Todas las figuras construidas correctamente.')

In [ ]:
# =====================================================================
# Generar el dashboard HTML
# =====================================================================

def fig_to_html_div(fig):
    return pio.to_html(fig, full_html=False, include_plotlyjs=False)

figs = {
    'kpi':    fig_kpi,
    'bar':    fig_bar2,
    'donut':  fig_donut,
    'line':   fig_line2,
    'sini':   fig_sini,
    'top5':   fig_top5,
    'prod':   fig_prod,
    'canal':  fig_canal,
}

divs = {k: fig_to_html_div(v) for k, v in figs.items()}

html_content = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Dashboard HDI Seguros Colombia</title>
  <script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{
      font-family: 'Segoe UI', Arial, sans-serif;
      background: #f4f6fb;
      color: #222;
    }}
    header {{
      background: linear-gradient(135deg, #003366 0%, #0055a5 100%);
      color: white;
      padding: 28px 40px;
      display: flex;
      align-items: center;
      gap: 20px;
      box-shadow: 0 2px 8px rgba(0,0,0,0.18);
    }}
    header h1 {{ font-size: 1.8rem; font-weight: 700; letter-spacing: 0.5px; }}
    header p  {{ font-size: 0.95rem; opacity: 0.85; margin-top: 4px; }}
    .badge {{
      background: #ffcc00; color: #003366;
      padding: 4px 12px; border-radius: 20px;
      font-weight: 700; font-size: 0.8rem; margin-left: auto;
    }}
    .container {{
      max-width: 1400px;
      margin: 0 auto;
      padding: 30px 24px;
    }}
    .section-title {{
      font-size: 1.1rem;
      font-weight: 700;
      color: #003366;
      margin: 32px 0 12px;
      padding-bottom: 6px;
      border-bottom: 3px solid #ffcc00;
      display: inline-block;
    }}
    .grid-2 {{
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 20px;
      margin-bottom: 20px;
    }}
    .grid-3 {{
      display: grid;
      grid-template-columns: 1fr 1fr 1fr;
      gap: 20px;
      margin-bottom: 20px;
    }}
    .full-width {{
      width: 100%;
      margin-bottom: 20px;
    }}
    .card {{
      background: white;
      border-radius: 12px;
      box-shadow: 0 2px 12px rgba(0,0,51,0.08);
      padding: 16px;
      overflow: hidden;
    }}
    .card .js-plotly-plot {{
      width: 100% !important;
    }}
    footer {{
      text-align: center;
      color: #888;
      font-size: 0.82rem;
      padding: 24px;
      border-top: 1px solid #dde;
      margin-top: 20px;
    }}
    @media (max-width: 900px) {{
      .grid-2, .grid-3 {{ grid-template-columns: 1fr; }}
    }}
  </style>
</head>
<body>

<header>
  <div>
    <h1>HDI Seguros Colombia — Dashboard de Producción</h1>
    <p>Análisis de pólizas, siniestros y primas · Datos 2023</p>
  </div>
  <span class="badge">Generado con Python · Plotly</span>
</header>

<div class="container">

  <h2 class="section-title">Resumen Ejecutivo</h2>
  <div class="full-width card">{divs['kpi']}</div>

  <h2 class="section-title">Tarea 1 &amp; 2 — Portafolio de Clientes y Pólizas</h2>
  <div class="grid-2">
    <div class="card">{divs['donut']}</div>
    <div class="card">{divs['prod']}</div>
  </div>

  <h2 class="section-title">Tarea 4 — Prima Emitida y Siniestralidad por Ramo</h2>
  <div class="grid-2">
    <div class="card">{divs['bar']}</div>
    <div class="card">{divs['sini']}</div>
  </div>

  <h2 class="section-title">Tarea 6 — Evolución Mensual Bogotá</h2>
  <div class="full-width card">{divs['line']}</div>

  <h2 class="section-title">Canal de Venta por Ciudad</h2>
  <div class="full-width card">{divs['canal']}</div>

  <h2 class="section-title">Tarea 5 — Top 5 Siniestros por Valor Aprobado</h2>
  <div class="full-width card">{divs['top5']}</div>

</div>

<footer>
  Dashboard generado automáticamente · HDI Seguros Colombia · 2024
  | Datos: clientes.csv · polizas.csv · siniestros.csv · primas_mensual.csv
</footer>

</body>
</html>
"""

output_html = Path('dashboard_hdi.html')
output_html.write_text(html_content, encoding='utf-8')
print(f'Dashboard generado: {output_html.resolve()}')
print(f'Tamaño: {output_html.stat().st_size / 1024:.1f} KB')